In [1]:
import spacy
from fastcoref import spacy_component
import requests

PORT = 8008
SERVER = "http://127.0.0.1"

podcasts = requests.get(f"{SERVER}:{PORT}/api/podcasts/")
podcasts = podcasts.json()
for i, p in enumerate(podcasts):
    print(f"{i}: {p['title']}")

/home/adamj/anaconda3/envs/whspr/lib/python3.8/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


0: Verdict with Ted Cruz
1: Ta Kommandoen med Geir Aker
2: Misjonen med Antonsen og Golden
3: Leger om livet
4: Norsken, svensken og dansken
5: Burde vært pensum
6: Huberman Lab
7: Stuff You Should Know
8: The Ben Shapiro Show
9: The Megyn Kelly Show
10: Pivot
11: The Daily
12: The Ezra Klein Show
13: Lex Fridman Podcast
14: Checks and Balance from The Economist
15: Money Talks from The Economist
16: The Economist Asks
17: The Ramsey Show
18: Dateline NBC
19: Pod Save America
20: Freakonomics Radio


In [2]:
import spacy
from fastcoref import spacy_component
import requests

for podcast in podcasts:
    if not podcast["language"].startswith("en"):
        print(f"skipping {podcast['title']}, language is {podcast['language']}")
        continue
    for audioitem in podcast['audioitem_set']:
        #if audioitem['title'] != "Ep. 1707 - World Famous YouTuber MrBeast Hit With Trans Controversy":
        #    continue
        print("starting episode: ", audioitem['title'], podcast['title'])
        for transcription in audioitem['transcription_set']:
            for segmentation in transcription['segmentation_set']:
                if segmentation['name'] == "spaCy":
                    seg_uuid = segmentation['uuid']
                    seg = requests.get(f"{SERVER}:{PORT}/api/segmentations/{seg_uuid}/")
                    seg = seg.json()
                    utterances = seg["utterance_set"]
                    # check if coref already exists in any of this segmentation's utterances
                    if any([utt["text_coref"] for utt in utterances]):
                        print("skipping ", audioitem['title'], podcast['title'])
                        continue
                    nlp = spacy.load("en_core_web_lg")
                    nlp.add_pipe(
                        "fastcoref", 
                        config={'model_architecture': 'LingMessCoref', 'model_path': 'biu-nlp/lingmess-coref', 'device': 'cuda:0'}
                    )

                    # add a context field to each utterance with the 50 previous utterances
                    for i, utt in enumerate(utterances):
                        utt["context"] = " ".join([u["text"] for u in utterances[max(0, i-50):i+1]])

                    docs = nlp.pipe(
                    [utterance["context"] for utterance in utterances],
                    component_cfg={"fastcoref": {'resolve_text': True}}
                    )
                    try:
                        docs = list(docs)
                    except:
                        print("error with ", audioitem['title'], podcast['title'])
                        for doc in docs:
                            print(doc)
                        continue

                    assert len(docs) == len(utterances)

                    for i, doc in enumerate(docs):
                        resolved_text = doc._.resolved_text

                        # Create a new Doc object without running the entire pipeline
                        sentences_doc = nlp.make_doc(resolved_text)

                        # Apply the "senter" component to the sentences_doc
                        nlp.get_pipe("senter")(sentences_doc)

                        first = coref = next(sentences_doc.sents)
                        for coref in sentences_doc.sents: pass

                        first = original = next(doc.sents)
                        for original in doc.sents: pass
                        if coref.text.strip() != original.text.strip():
                            # write to API
                            utt_uuid = utterances[i]["uuid"]
                            utterances[i]["text_coref"] = coref.text.strip()
                            # no changes to these child record, so remove them
                            utterances[i].pop("classification_set")
                            utterances[i].pop("query_set")
                            res = requests.post(f"{SERVER}:{PORT}/api/utterances/{utt_uuid}/", json=utterances[i])
                            print(res.status_code)

                    

starting episode:  Trump Pleads Not Guilty - What Happens Now? Verdict with Ted Cruz
skipping  Trump Pleads Not Guilty - What Happens Now? Verdict with Ted Cruz
starting episode:  Trump Indictment - What Are Trump's Options Legally, To Fight Back! Verdict with Ted Cruz
skipping  Trump Indictment - What Are Trump's Options Legally, To Fight Back! Verdict with Ted Cruz
starting episode:  Trump Indictment - What It Means & What's Next Verdict with Ted Cruz
skipping  Trump Indictment - What It Means & What's Next Verdict with Ted Cruz
starting episode:  How Can We STOP School Shootings - and, Maddeningly, Why Democrats Keep Blocking Real School Safety Legislation Verdict with Ted Cruz
skipping  How Can We STOP School Shootings - and, Maddeningly, Why Democrats Keep Blocking Real School Safety Legislation Verdict with Ted Cruz
starting episode:  Mayorkas LIED: Explosive Cross-Examination In The Senate Judiciary Committee Verdict with Ted Cruz
skipping  Mayorkas LIED: Explosive Cross-Examina

Some weights of the model checkpoint at biu-nlp/lingmess-coref were not used when initializing LingMessModel: ['longformer.embeddings.position_ids']
- This IS expected if you are initializing LingMessModel from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing LingMessModel from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
05/06/2023 00:16:58 - INFO - 	 missing_keys: []
05/06/2023 00:16:58 - INFO - 	 unexpected_keys: []
05/06/2023 00:16:58 - INFO - 	 mismatched_keys: []
05/06/2023 00:16:58 - INFO - 	 error_msgs: []
05/06/2023 00:16:58 - INFO - 	 Model Parameters: 590.0M, Transformer: 434.6M, Coref head: 155.4M
05/06/2023 00:17:22 - INFO - 	 Tokenize 256 inputs...
05/06/2023 00:17:32 - INFO - 	 ***** Runnin

200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
starting episode:  Manhunt Underway for Illegal Alien Who Committed Mass Murder, plus More Damning News About Hunter Biden's

05/06/2023 00:18:58 - INFO - 	 missing_keys: []
05/06/2023 00:18:58 - INFO - 	 unexpected_keys: []
05/06/2023 00:18:58 - INFO - 	 mismatched_keys: []
05/06/2023 00:18:58 - INFO - 	 error_msgs: []
05/06/2023 00:18:58 - INFO - 	 Model Parameters: 590.0M, Transformer: 434.6M, Coref head: 155.4M
05/06/2023 00:19:12 - INFO - 	 Tokenize 256 inputs...
05/06/2023 00:19:21 - INFO - 	 ***** Running Inference on 256 texts *****
Inference: 100%|██████████| 256/256 [00:25<00:00, 10.08it/s]
05/06/2023 00:19:57 - INFO - 	 Tokenize 180 inputs...
05/06/2023 00:20:01 - INFO - 	 ***** Running Inference on 180 texts *****
Inference: 100%|██████████| 180/180 [00:17<00:00, 10.31it/s]


200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200


...

error with  Why Are There So Many Bad Bosses? (Ep. 495 Replay) Freakonomics Radio
starting episode:  536. Is Your Plane Ticket Too Expensive — or Too Cheap? Freakonomics Radio
skipping  536. Is Your Plane Ticket Too Expensive — or Too Cheap? Freakonomics Radio
starting episode:  535. Why Is Flying Safer Than Driving? Freakonomics Radio
skipping  535. Why Is Flying Safer Than Driving? Freakonomics Radio
starting episode:  538. A Radically Simple Way to Boost a Neighborhood Freakonomics Radio
skipping  538. A Radically Simple Way to Boost a Neighborhood Freakonomics Radio
starting episode:  539. Why Does One Tiny State Set the Rules for Everyone? Freakonomics Radio


05/06/2023 02:26:14 - INFO - 	 missing_keys: []
05/06/2023 02:26:14 - INFO - 	 unexpected_keys: []
05/06/2023 02:26:14 - INFO - 	 mismatched_keys: []
05/06/2023 02:26:14 - INFO - 	 error_msgs: []
05/06/2023 02:26:14 - INFO - 	 Model Parameters: 590.0M, Transformer: 434.6M, Coref head: 155.4M
05/06/2023 02:26:24 - INFO - 	 Tokenize 256 inputs...
05/06/2023 02:26:30 - INFO - 	 ***** Running Inference on 256 texts *****
Inference: 100%|██████████| 256/256 [00:19<00:00, 13.22it/s]


error with  539. Why Does One Tiny State Set the Rules for Everyone? Freakonomics Radio
starting episode:  Why Your Projects Are Always Late — and What to Do About It (Ep. 323 Replay) Freakonomics Radio
skipping  Why Your Projects Are Always Late — and What to Do About It (Ep. 323 Replay) Freakonomics Radio
starting episode:  540. Swearing Is More Important Than You Think Freakonomics Radio


05/06/2023 02:26:54 - INFO - 	 missing_keys: []
05/06/2023 02:26:54 - INFO - 	 unexpected_keys: []
05/06/2023 02:26:54 - INFO - 	 mismatched_keys: []
05/06/2023 02:26:54 - INFO - 	 error_msgs: []
05/06/2023 02:26:54 - INFO - 	 Model Parameters: 590.0M, Transformer: 434.6M, Coref head: 155.4M
05/06/2023 02:27:03 - INFO - 	 Tokenize 256 inputs...
05/06/2023 02:27:07 - INFO - 	 ***** Running Inference on 256 texts *****
Inference: 100%|██████████| 256/256 [00:17<00:00, 14.72it/s]
05/06/2023 02:27:35 - INFO - 	 Tokenize 256 inputs...
05/06/2023 02:27:40 - INFO - 	 ***** Running Inference on 256 texts *****
Inference: 100%|██████████| 256/256 [00:21<00:00, 12.00it/s]

error with  540. Swearing Is More Important Than You Think Freakonomics Radio
starting episode:  541. The Case of the $4 Million Gold Coffin Freakonomics Radio
